## Imports

In [85]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

import warnings
warnings.filterwarnings("ignore")

In [86]:
from funcoes_escoragem import *

## Diretório

In [87]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Google BigQuery

In [88]:
project_id = 'loft-dl-fintech'

# CredPago - Consulta Realizada**

In [89]:
query_credpago = f"""
WITH consulta_realizada AS (
    SELECT
        CAST(REGEXP_REPLACE(documento, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
        id_externo AS contract_id,

        MIN(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS requested_at,

        MAX(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS data_ultima_consulta,
        ARRAY_LENGTH(JSON_EXTRACT_ARRAY(CR.json_retornado, '$.pessoas')) AS qtd_proponentes,
        CR.*
    FROM `loft-dl-fintech.bronze_credpago_sortinghat.consulta_realizada` CR
    WHERE DATE(data) >= DATE_SUB(CURRENT_DATE(), INTERVAL 4 WEEK)
    AND DATE(data) <= CURRENT_DATE()
)

SELECT *
FROM consulta_realizada
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY 
        CASE WHEN model = 'BLEND_4' THEN 1 ELSE 2 END ASC,
        data DESC
) = 1
"""

credpago_df = pd.read_gbq(query_credpago, project_id=project_id)
credpago_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,renda_considerada,model,id_externo,json_retornado,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result
0,41147288801,4119388,2026-06-22,2026-06-22,1,B,9247.500000000,BLEND3_3,4119388,"{""pessoas"":[{""nome"":""DANIEL POLONIO FRANCISCO""...",...,31,1778785248777,"{""valor_aluguel"":""4924.94"",""imobiliaria_id"":30...",2026-06-22 18:00:27+00:00,1782151219060959648,41147288801,A,2026-06-22 18:11:04.096000+00:00,2026-06-22 08:55:54+00:00,APROVAR
1,52164953800,4164328,2026-06-12,2026-06-12,1,NI,1507.000000000,BLEND3_3,4164328,"{""pessoas"":[{""nome"":""RAPHAELLA THELES SILVA RO...",...,33,1778785248777,"{""valor_aluguel"":""1200"",""imobiliaria_id"":50799...",2026-06-13 08:00:40+00:00,1781337639406946309,52164953800,E,2026-06-13 08:05:45.751000+00:00,2026-06-12 18:07:03+00:00,APROVAR
2,44943642829,4167314,2026-06-15,2026-06-15,1,C,2808.500000000,BLEND3_3,4167314,"{""pessoas"":[{""nome"":""BIANCA FINOTELLO DA SILVA...",...,33,1778785248777,"{""valor_aluguel"":""3800"",""imobiliaria_id"":9378,...",2026-06-15 18:00:36+00:00,1781546428411576996,44943642829,E,2026-06-15 18:01:18.049000+00:00,2026-06-15 07:57:58+00:00,REPROVAR
3,56879911072,4179123,2026-06-23,2026-06-23,1,E,9453.000000000,BLEND3_3,4179123,"{""pessoas"":[{""nome"":""ISALTINA SILVEIRA ESPINOS...",...,34,1778785248777,"{""valor_aluguel"":""1500"",""imobiliaria_id"":9570,...",2026-06-24 08:00:33+00:00,1782288029574372302,56879911072,C,2026-06-24 08:06:51.681000+00:00,2026-06-23 15:18:46+00:00,DERIVAR
4,5779134502,4189335,2026-06-23,2026-06-23,1,B,7535.000000000,BLEND_REGRESSAO_2026,4189335,"{""pessoas"":[{""nome"":""FELIPE CARVALHO ROCHA"",""d...",...,36,1778785248777,"{""valor_aluguel"":""1200"",""imobiliaria_id"":592,""...",2026-06-23 18:00:44+00:00,1782237643464570557,05779134502,D,2026-06-23 18:10:22.710000+00:00,2026-06-23 14:43:00+00:00,APROVAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110463,86978113506,4354657,2026-07-09,2026-07-09,1,A,2877.000000000,BLEND3_3,4354657,"{""pessoas"":[{""nome"":""FERNANDO SEGUNDO LASTRE M...",...,34,1778785248777,"{""valor_aluguel"":1600,""matchmaking_on"":false,""...",2026-07-10 08:00:44+00:00,1783670442334933785,86978113506,C,2026-07-10 08:11:21.007000+00:00,2026-07-09 17:17:31+00:00,APROVAR
110464,3251842080,4354681,2026-07-09,2026-07-09,1,NI,2603.000000000,BLEND3_3,4354681,"{""pessoas"":[{""nome"":""EVERSON ROSIEL DOS SANTOS...",...,34,1778785248777,"{""valor_aluguel"":1500,""matchmaking_on"":false,""...",2026-07-10 08:00:44+00:00,1783670442380007425,03251842080,C,2026-07-10 08:11:21.017000+00:00,2026-07-09 17:20:06+00:00,APROVAR
110465,2811305904,4354719,2026-07-09,2026-07-09,1,C,5069.000000000,BLEND3_3,4354719,"{""pessoas"":[{""nome"":""ERIVELTON ANDERSON PEREIR...",...,34,1778785248777,"{""valor_aluguel"":1800,""matchmaking_on"":false,""...",2026-07-10 08:00:44+00:00,1783670442425772623,02811305904,C,2026-07-10 08:11:21.035000+00:00,2026-07-09 17:26:43+00:00,APROVAR
110466,5290711775,4354915,2026-07-09,2026-07-09,1,NI,40004.000000000,BLEND3_3,4354915,"{""pessoas"":[{""nome"":""ANDRESSA RODRIGUES BRAGA ...",...,31,1778785248777,"{""valor_aluguel"":1600,""matchmaking_on"":false,""...",2026-07-10 08:00:44+00:00,1783670442767213655,05290711775,A,2026-07-10 08:11:21.127000+00:00,2026-07-09 18:05:47+00:00,APROVAR


In [90]:
    # ORDER BY 
    #     CASE WHEN model = 'BLEND_4' THEN 1 ELSE 2 END ASC,
    #     data DESC

In [91]:
mask = credpago_df['contract_id'].astype(str).str.match(r'^\d+$')
credpago_df = credpago_df[mask].copy()
credpago_df['contract_id'] = credpago_df['contract_id'].astype(int)

In [92]:
credpago_df['model'].value_counts(dropna=False)

model
BLEND3_3                94294
BLEND_REGRESSAO_2026     6897
HVA3                     4413
BLEND_4                  2518
BVS_CUSTOM               1522
HVA4                      708
BVS_CUSTOM_V2              93
                           15
HFT1                        8
Name: count, dtype: int64

## Multiproponente

In [93]:
credpago_df['qtd_proponentes'].value_counts(dropna=False)

qtd_proponentes
1    106139
2      3985
3       315
4        26
8         1
6         1
5         1
Name: count, dtype: Int64

In [94]:
credpago_df['qtd_proponentes'].value_counts(normalize=True, dropna=False)

qtd_proponentes
1    0.960812
2    0.036074
3    0.002852
4    0.000235
8    0.000009
6    0.000009
5    0.000009
Name: proportion, dtype: Float64

## Quebrar JSON - Retornado

In [95]:
def unwrap_payload(obj):
    """Supports old format (message wrapper) and new format (root payload)."""
    if not obj:
        return None
    if isinstance(obj, dict) and "message" in obj:
        return obj["message"]
    return obj

def get_pessoas(obj):
    payload = unwrap_payload(obj)
    return (payload or {}).get("pessoas") or []

In [96]:
parsed = credpago_df["json_retornado"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

valid_parsed = parsed.dropna()
payload_parsed = valid_parsed.apply(unwrap_payload)

# json_normalize resets the index; preserve credpago_df labels for the join below
contrato_df = pd.json_normalize(payload_parsed.tolist(), sep="_")
contrato_df.index = payload_parsed.index
contrato_df = contrato_df.add_prefix("message_")  # keeps message_decisao, message_scores_*, etc.

pessoa_records = payload_parsed.apply(lambda x: (get_pessoas(x) or [{}])[0])
pessoa_df = pd.json_normalize(pessoa_records.tolist(), sep="_")
pessoa_df.index = payload_parsed.index
pessoa_df = pessoa_df.add_prefix("pessoa_")

In [97]:
valid = parsed.notna()
base_idx = credpago_df.loc[valid].index

pendencias = []
for idx, row in parsed[valid].items():
    for p in get_pessoas(row):
        if not isinstance(p, dict):
            continue

        serasa = p.get("anotacoesFinanceirasSerasa") or {}
        pend_list = serasa.get("PendenciasPagamentoPF") or []

        for pend in pend_list:
            if isinstance(pend, dict):
                pendencias.append({"idx": idx, **pend})

pendencias_df = pd.DataFrame(pendencias)

if not pendencias_df.empty:
    pendencias_agg = (
        pendencias_df
        .groupby("idx", as_index=False)
        .agg(
            qtde_pefin=("Valor", "count"),
            valor_pefin_total=("Valor", lambda s: pd.to_numeric(s, errors="coerce").sum()),
            modalidades_pefin=("Modalidade", lambda s: ",".join(sorted(set(s.dropna().astype(str))))),
        )
    )
else:
    pendencias_agg = pd.DataFrame(columns=["idx", "qtde_pefin", "valor_pefin_total", "modalidades_pefin"])

## Quebrar JSON - Request


In [98]:
request_parsed = credpago_df["request"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

request_valid = request_parsed.dropna()
request_df = pd.json_normalize(request_valid.tolist(), sep="_")
request_df.index = request_valid.index
request_df = request_df.add_prefix("request_")

# Unify old (snake_case) and new (camelCase) request schemas
REQUEST_FIELD_ALIASES = {
    "request_valorAluguel": ["request_valor_aluguel"],
    "request_imobiliariaId": ["request_imobiliaria_id"],
    "request_idExterno": ["request_id_externo"],
    "request_imovelTipo": ["request_imovel_tipo"],
    "request_principalDocument": ["request_pessoa_principal_documento"],
}

for target, sources in REQUEST_FIELD_ALIASES.items():
    existing_sources = [c for c in sources if c in request_df.columns]
    if not existing_sources and target not in request_df.columns:
        continue
    if target not in request_df.columns:
        request_df[target] = pd.NA
    for source in existing_sources:
        if target == "request_valorAluguel":
            request_df[target] = request_df[target].combine_first(
                pd.to_numeric(request_df[source], errors="coerce")
            )
        else:
            request_df[target] = request_df[target].combine_first(request_df[source])
        request_df = request_df.drop(columns=[source])


## Join Json's

In [99]:
EXPANDED_PREFIXES = ("message_", "pessoa_", "request_")
expanded_cols = [c for c in credpago_df.columns if c.startswith(EXPANDED_PREFIXES)]

base_df = credpago_df.loc[valid].drop(columns=expanded_cols, errors="ignore")

credpago_expandido = (
    base_df
    .join(contrato_df, how="left")
    .join(pessoa_df, how="left")
    .join(request_df, how="left")
    .reset_index(names="idx")
    .merge(pendencias_agg, on="idx", how="left")
    .drop(columns="idx")
)


In [100]:
cols_drop = [
    # ingestão
    "_sdc_table_version", "_sdc_received_at", "_sdc_sequence", "_sdc_batched_at",

    # json bruto / containers
    "json_retornado",
    "request",
    "message_pessoas", "message_scores", "message_ratings",
    "pessoa_scores", "pessoa_ratings", "message_blend3Variables",
    "request_priorDecisions", "request_dadosAusentes", "request_errosTecnicos",
    "request_outras_pessoas", "request_blend3Variables", "request_scores",

    # bronze redundante
    "score_bvs",
    "renda_considerada",
    "decisao_bureau",
    "limite_mensal",

    # prováveis duplicatas
    "id_externo",
    "data",
    "request_month",

    # operacional / baixo valor (old format only — safe to keep)
    "success", "message_manual", "message_node",
    "message_reaproveitamentoConsultaMotor", "message_savingBureauPowerCurve",
    "message_logs", "message_partnerIds",
    "message_errosTecnicos", "message_dadosAusentes",
    "pessoa_errosTecnicos", "pessoa_dadosAusentes",
    "message_rentGuaranteeConstraints_rent_coverage_multiplier",
    "message_rentGuaranteeConstraints_exit_cost_multiplier",
    "message_rentGuaranteeConstraints_commission_percent",
    "message_rentGuaranteeConstraints_version",
]

credpago_clean = credpago_expandido.drop(columns=[c for c in cols_drop if c in credpago_expandido.columns])


In [101]:
credpago_clean[
    (pd.to_datetime(credpago_clean["requested_at"]).dt.normalize() == "2026-07-03")
    & (credpago_clean["message_decisao"] == "BLEND_4")
][["message_blend3Variables_realEstatePc4HistoryOver100Contracts", "message_blend3Variables_cityPc4HistoryOver100Contracts"]]

,message_blend3Variables_realEstatePc4HistoryOver100Contracts,message_blend3Variables_cityPc4HistoryOver100Contracts
1910,0.000000,0.064516
1912,0.000000,0.086614
1935,0.000000,0.106942
3005,0.000000,0.104675
3007,0.000000,0.130864
...,...,...
108012,NaN,0.072148
109105,0.000000,0.072189
110214,0.142857,0.094241
110215,0.235465,0.147808


## Escoragem Blend4

In [102]:
credpago_clean_rep = credpago_clean.replace({
    "request_imovelTipo": {"RESIDENCIAL": 1, "COMERCIAL": 0}, 
    "message_blend3Variables_hasPreviousContracts": {True: 1, False: 0},
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": {True: 1, False: 0}
}
)

In [103]:
rename_dict = {
    "message_scores_BVS_CUSTOM_V2": "score_proposto__bvs",#
    "message_scores_HFT1": "SERASA_CHSV5",
    "pessoa_idade": "age",
    "request_imovelTipo": "property_type",
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "request_valorAluguel": "rental_value",
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [104]:
vars_blend4 = ['score_proposto__bvs', 'SERASA_CHSV5', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [105]:
df_predict = (credpago_clean_rep.copy()).rename(columns=rename_dict)
df_predict["REGRA_BLEND_4"] = np.where(
    df_predict["score_proposto__bvs"] < 334,
    "E_BVS",
    "BLEND4",
)

df_predict["SCORE_BVS"] = df_predict["score_proposto__bvs"]
df_predict["SCORE_SERASA"] = df_predict["SERASA_CHSV5"]
df_predict["RENDA"] = df_predict["income"]

df_predict.head()

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,request_blend3Variables_hadOverdueInvoiceInPreviousContracts,request_blend3Variables_cityPc4HistoryOver100Contracts,request_blend3Variables_realEstatePc4HistoryOver100Contracts,qtde_pefin,valor_pefin_total,modalidades_pefin,REGRA_BLEND_4,SCORE_BVS,SCORE_SERASA,RENDA
0,41147288801,4119388,2026-06-22,2026-06-22,1,B,BLEND3_3,5979703,31,41147288801,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,9247.5
1,52164953800,4164328,2026-06-12,2026-06-12,1,NI,BLEND3_3,5942832,33,52164953800,...,NaN,NaN,NaN,2.0,3388.0,"CRED CARTAO,FINANCIAMENT",BLEND4,NaN,NaN,1507.0
2,44943642829,4167314,2026-06-15,2026-06-15,1,C,BLEND3_3,5944261,33,44943642829,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,2808.5
3,56879911072,4179123,2026-06-23,2026-06-23,1,E,BLEND3_3,5992142,34,56879911072,...,NaN,NaN,NaN,2.0,79010.0,"CRED CARTAO,EMPRESTIMO",BLEND4,NaN,NaN,9453.0
4,5779134502,4189335,2026-06-23,2026-06-23,1,B,BLEND_REGRESSAO_2026,5991678,36,05779134502,...,NaN,NaN,NaN,3.0,3841.0,"CONS VEICULO,CRED CARTAO",BLEND4,NaN,NaN,7535.0


In [106]:
df_predict.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    110224
E_BVS        244
dtype: int64

In [107]:
# df_predict.to_csv(ANALYTICS_DIR/"df_predict_blend4_pre.csv", index=False)

## Criacão das Variáveis

In [108]:
df_predict_vars = prepare_blend4_variables(df_predict)
df_predict_escorada = predict_blend4_1(df_predict_vars)

## Criar Rating Blend4

In [109]:
score = pd.to_numeric(df_predict_escorada["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_manual_blend4"] = np.select(conditions, choices, default=None)

In [110]:
score = pd.to_numeric(df_predict_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_json_blend4"] = np.select(conditions, choices, default=None)

## Salvar Base como se fosse Blend4

In [111]:
# df_predict_blend4 = df_predict_escorada[df_predict_escorada["message_decisao"] == "BLEND3_3"].copy()
# df_predict_blend4["message_decisao"] = df_predict_blend4["message_decisao"].replace("BLEND3_3", "BLEND4")

# cp_final_df = pd.concat([df_predict_escorada, df_predict_blend4])
cp_final_df = df_predict_escorada.copy()
cp_final_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,SERASA_CHSV5__normalized4_1,age__normalized4_1,qtde_restricoes__consulta_realizada__normalized4_1,income_commitment__normalized4_1,agency_pc4_mais_100_contratos__pc_categorias__normalized4_1,city_pc4_mais_100_contratos__pc_categorias__normalized4_1,pred_blend4_1,pred_blend4_1_to_score,rating_manual_blend4,rating_json_blend4
0,41147288801,4119388,2026-06-22,2026-06-22,1,B,BLEND3_3,5979703,31,41147288801,...,0.0,-0.15,0.0,0.226688,0.0,-0.128057,0.524099,476.0,B,A
1,52164953800,4164328,2026-06-12,2026-06-12,1,NI,BLEND3_3,5942832,33,52164953800,...,0.0,-0.65,2.0,0.809205,0.0,-0.128057,0.540801,459.0,B,E
2,44943642829,4167314,2026-06-15,2026-06-15,1,C,BLEND3_3,5944261,33,44943642829,...,0.0,-0.20,0.0,2.039012,0.0,0.174016,0.564454,436.0,C,E
3,56879911072,4179123,2026-06-23,2026-06-23,1,E,BLEND3_3,5992142,34,56879911072,...,0.0,1.55,2.0,-0.599196,0.0,-0.523287,0.479791,520.0,A,A
4,5779134502,4189335,2026-06-23,2026-06-23,1,B,BLEND_REGRESSAO_2026,5991678,36,05779134502,...,0.0,0.00,3.0,-0.597922,0.0,0.000000,0.499217,501.0,A,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110463,86978113506,4354657,2026-07-09,2026-07-09,1,A,BLEND3_3,6074317,34,86978113506,...,0.0,1.35,0.0,0.278741,0.0,-0.578440,0.483517,516.0,A,A
110464,3251842080,4354681,2026-07-09,2026-07-09,1,NI,BLEND3_3,6074343,34,03251842080,...,0.0,0.00,0.0,0.323191,0.0,0.000000,0.498263,502.0,A,C
110465,2811305904,4354719,2026-07-09,2026-07-09,1,C,BLEND3_3,6074392,34,02811305904,...,0.0,0.65,0.0,-0.165325,0.0,-1.001829,0.471657,528.0,A,A
110466,5290711775,4354915,2026-07-09,2026-07-09,1,NI,BLEND3_3,6074634,31,05290711775,...,0.0,0.55,0.0,-0.861357,0.0,-1.178492,0.449490,551.0,A,A


In [112]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                           15
BLEND3_3                94294
BLEND_4                  2518
BLEND_REGRESSAO_2026     6897
BVS_CUSTOM               1522
BVS_CUSTOM_V2              93
HFT1                        8
HVA3                     4413
HVA4                      708
dtype: int64

In [113]:
bvs = pd.to_numeric(cp_final_df["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(cp_final_df["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.E",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

cp_final_df["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

## Salvar

In [114]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    110224
E_BVS        244
dtype: int64

In [115]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                           15
BLEND3_3                94294
BLEND_4                  2518
BLEND_REGRESSAO_2026     6897
BVS_CUSTOM               1522
BVS_CUSTOM_V2              93
HFT1                        8
HVA3                     4413
HVA4                      708
dtype: int64

In [116]:
cp_final_df.groupby("model", dropna=False).size()

model
                           15
BLEND3_3                94294
BLEND_4                  2518
BLEND_REGRESSAO_2026     6897
BVS_CUSTOM               1522
BVS_CUSTOM_V2              93
HFT1                        8
HVA3                     4413
HVA4                      708
dtype: int64

In [117]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    110224
E_BVS        244
dtype: int64

In [118]:
cp_final_df.to_csv(ANALYTICS_DIR/"df_predict_blend4.csv", index=False)

## Investigação Sexta-Feira (03/07)

In [119]:
cp_final_df[
    (pd.to_datetime(cp_final_df["requested_at"]).dt.normalize() == "2026-07-03")
    & (cp_final_df["message_decisao"] == "BLEND_4")
    & (cp_final_df["pred_blend4_1_to_score"] != cp_final_df["message_blendRegressaoPredict"])
][id_vars + vars_blend4 + ["pred_blend4_1_to_score", "message_blendRegressaoPredict"]]

,contract_id,requested_at,CPF_CNPJ,message_decisao,message_blendRegressaoPredict,score_proposto__bvs,SERASA_CHSV5,age,property_type,qtde_restricoes__consulta_realizada,rental_value,income,agency_pc4_mais_100_contratos__pc_categorias,city_pc4_mais_100_contratos__pc_categorias,flag_tem__contratos_anteriores,flag_teve_boleto_atrasado__contratos_anteriores,pred_blend4_1_to_score,message_blendRegressaoPredict
1910,4329751,2026-07-03,51290576866,BLEND_4,646.0,593.0,644.0,24.0,1,0,2500.0,3014.0,0.0,0.064516,0.0,0.0,614.0,646.0
1912,4329834,2026-07-03,2556934948,BLEND_4,387.0,440.0,425.0,47.0,1,5,5500.0,6987.0,0.0,0.086614,0.0,0.0,356.0,387.0
1935,4332406,2026-07-03,8011979884,BLEND_4,365.0,491.0,469.0,64.0,0,12,3200.0,8905.0,0.0,0.106942,1.0,1.0,334.0,365.0
3005,4331023,2026-07-03,1615508945,BLEND_4,421.0,409.0,447.0,64.0,1,0,3800.0,15275.5,0.0,0.104675,0.0,0.0,388.0,421.0
3007,4331076,2026-07-03,7569622936,BLEND_4,655.0,693.0,560.0,33.0,1,2,2300.0,21509.0,0.0,0.130864,1.0,0.0,625.0,655.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105781,4329890,2026-07-03,46224044864,BLEND_4,587.0,578.0,571.0,26.0,1,0,1300.0,1438.5,0.0,0.000000,0.0,0.0,628.0,587.0
105793,4331786,2026-07-03,27805930910,BLEND_4,526.0,524.0,572.0,70.0,1,0,2300.0,2055.0,0.0,0.107708,1.0,0.0,492.0,526.0
106896,4332143,2026-07-03,56694393091,BLEND_4,726.0,616.0,787.0,58.0,1,0,4900.0,32195.0,0.0,0.087567,0.0,0.0,699.0,726.0
109105,4331846,2026-07-03,5359650958,BLEND_4,283.0,374.0,166.0,35.0,1,3,1800.0,3904.5,0.0,0.072189,0.0,0.0,257.0,283.0
